In [26]:
import json
import os
import uuid 
from typing import Optional
from typing import List


We used uuid library to create unique user identification (extremely helpful in real life scenario for users with same name).

to_dict() will help save user data as JSON since JSON can't store python objects.

from_dict() lets us load JSON back into python user objects for easy access.

I applied .strip().lower() to "preferred_environment" for easier matching later and avoid in code errors.

In [27]:
class User:
    def __init__(self, name: str, group_size: int, preferred_environment: str, budget_min: float, 
                 budget_max: float, user_id: Optional[str] = None):
    
        self.user_id = user_id if user_id else str(uuid.uuid4())
        self.name = name.strip()
        self.group_size = group_size
        self.preferred_environment = preferred_environment.strip().lower()
        self.budget_min = budget_min
        self.budget_max = budget_max
    
    # Converting User object to Dictionary to save as JSON
    def to_dict(self):
        return {
            "user_id": self.user_id,
            "name": self.name,
            "group_size": self.group_size,
            "preferred_environment": self.preferred_environment,
            "budget_min": self.budget_min,
            "budget_max": self.budget_max
        }
    
    # Creating a User object from a dictionary (loaded from JSON)
    @staticmethod
    def from_dict(data: dict):
        return User(
            user_id=data.get("user_id"),
            name=data["name"],
            group_size=data["group_size"],
            preferred_environment=data["preferred_environment"],
            budget_min=data["budget_min"],
            budget_max=data["budget_max"]
        )

user_dicts initializes an empty list to hold user data in dictionary form.

After loopiing via every user in the list, user object is converted to a dictionary using its to_dict() method, then added to user_dicts.

The file is opened with a write permission and then a list of user dictionaries is written to the file in JSON format with good indentation.

-----------------------------------------------------------------------------------------------------------------------------------------------

The "-> List[User]" is a type hint showing the return type.

load_users function will open the file and load the JSON content (list of dictionaries) from the file into user_dicts.

Then the dictionary back into a user object using the from_dict static method.

-----------------------------------------------------------------------------------------------------------------------------------------------

Saving users:
When the app wants to save all users, it takes the list of User objects, converts each to a dictionary, then saves the entire list as JSON to a file. This way, user data is saved on disk between program runs.

Loading users:
When the app needs user data, it tries to open the JSON file, reads the list of dictionaries, converts each back into a User object, and returns this list. If no file exists or the file is corrupted, it safely returns an empty list so the app can continue working without errors.

In [28]:
USERS_FILE = '/Users/ishaandawra/Desktop/Machine Learning Notes/Machine Learning Projects/LLM_SummerHome_Recommender/src/users.json'

def save_users(users: List[User], filename=USERS_FILE):

    user_dicts = []
    for user in users:
        user_dicts.append(user.to_dict())
        
    with open(filename, 'w') as f:
        json.dump(user_dicts, f, indent=4)

    print(f"Saved {len(users)} user profiles to {filename}.")


def load_users(filename=USERS_FILE) -> List[User]:
    try:
        with open(filename, 'r') as f:
            user_dicts = json.load(f)
        
        users = []
        for user_data in user_dicts:
            user = User.from_dict(user_data)
            users.append(user)
        
        print(f"Loaded {len(users)} user profiles from {filename}.")
        return users
    
    except FileNotFoundError:
        print(f"No user file found at {filename}. Starting with empty list.")
        return []
    except json.JSONDecodeError:
        print(f"User file at {filename} is empty or corrupted. Starting fresh.")
        return []


In [ ]:
# CRUD Operations for User Management - Create, Read, Update, Delete

def create_user(users: List[User], name: str, group_size: int, preferred_environment: str, 
                budget_min: float, budget_max: float) -> User:
    
    new_user = User(
        name=name,
        group_size=group_size,
        preferred_environment=preferred_environment,
        budget_min=budget_min,
        budget_max=budget_max,
        user_id=None  
    )
    users.append(new_user)
    save_users(users)  
    return new_user



def view_users(users: List[User]) -> None:

    if not users:
        print("No users found.")
        return
    
    for user in users:
        print(f"User_ID: {user.user_id} | Name: {user.name} | Group Size: {user.group_size} | "
              f"Environment: {user.preferred_environment} | Budget: {user.budget_min}-{user.budget_max}")




def find_user_by_id(users: List[User], user_id: str) -> Optional[User]:

    for user in users:
        if user.user_id == user_id:
            return user
    return None



def update_user(users, user_id, name=None, group_size=None, preferred_environment=None, budget_min=None, budget_max=None):

    user = find_user_by_id(users, user_id)
    if not user:
        return False
    
    if name is not None:
        user.name = name
    if group_size is not None:
        user.group_size = group_size
    if preferred_environment is not None:
        user.preferred_environment = preferred_environment
    if budget_min is not None:
        user.budget_min = budget_min
    if budget_max is not None:
        user.budget_max = budget_max
    
    save_users(users)
    return True



def delete_user(users: List[User], user_id: str) -> bool:
 
    for i, user in enumerate(users): # I used enumerate to get index of user while iterating along with user_id.
        if user.user_id == user_id:
            users.pop(i)
            save_users(users)
            return True
    return False

Manual Testing (I hope everything works good !!)

In [67]:
def print_all_users(users):
    if not users:
        print("No users found.")
        return
    print(f"Total users: {len(users)}")
    for user in users:
        print(f"User ID: {user.user_id}")
        print(f"Name: {user.name}")
        print(f"Group Size: {user.group_size}")
        print(f"Preferred Environment: {user.preferred_environment}")
        print(f"Budget: ${user.budget_min} to ${user.budget_max}")
        print("-" * 40)

In [68]:
users = load_users()
print_all_users(users)

User file at /Users/ishaandawra/Desktop/Machine Learning Notes/Machine Learning Projects/LLM_SummerHome_Recommender/src/users.json is empty or corrupted. Starting fresh.
No users found.


In [69]:
new_user = User(
    name="Alice",
    group_size=3,
    preferred_environment="beach",
    budget_min=100,
    budget_max=300
)

users.append(new_user)
save_users(users)
print("\n After adding new user:")
print_all_users(users)

Saved 1 user profiles to /Users/ishaandawra/Desktop/Machine Learning Notes/Machine Learning Projects/LLM_SummerHome_Recommender/src/users.json.

 After adding new user:
Total users: 1
User ID: 1b88b951-3fea-4f79-983e-ab8bece1380a
Name: Alice
Group Size: 3
Preferred Environment: beach
Budget: $100 to $300
----------------------------------------


In [70]:
update_success = update_user(users, new_user.user_id, budget_max=1000, preferred_environment="mountain")
if update_success:
    print("\n After updating user details:")
    print_all_users(users)
else:
    print("Update failed")

Saved 1 user profiles to /Users/ishaandawra/Desktop/Machine Learning Notes/Machine Learning Projects/LLM_SummerHome_Recommender/src/users.json.

 After updating user details:
Total users: 1
User ID: 1b88b951-3fea-4f79-983e-ab8bece1380a
Name: Alice
Group Size: 3
Preferred Environment: mountain
Budget: $100 to $1000
----------------------------------------


In [ ]:
# delete_success = delete_user(users, new_user.user_id)
# if delete_success:
#     print("\nAfter deleting user:")
#     print_all_users(users)
# else:
#     print("Delete failed: User not found")

Saved 0 user profiles to /Users/ishaandawra/Desktop/Machine Learning Notes/Machine Learning Projects/LLM_SummerHome_Recommender/src/users.json.

After deleting user:
No users found.
